# microgpt 教程

本文翻译自 https://karpathy.github.io/2026/02/12/microgpt/

<br />

仅用200行纯Python代码（零依赖），即可训练并推理GPT——这已经是我能压缩到的极致了。

2026-02-12 07:00:00

---

这是我的新艺术项目 [microgpt](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95) 的简要指南。一个200行的纯Python单文件，无任何依赖，能够训练和推理GPT。该文件包含了所需的完整算法内容：文档数据集、分词器、自动微分引擎、GPT-2风格的神经网络架构、Adam优化器、训练循环和推理循环。其他一切都只是效率问题。这个脚本是多个项目（micrograd、makemore、nanogpt等）以及十年执着于将LLM简化到其本质的结晶。

相关链接：

- GitHub Gist 完整源代码：[microgpt.py](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95)
- 网页版：[https://karpathy.ai/microgpt.html](https://karpathy.ai/microgpt.html)
- Google Colab 笔记本：[点击打开](https://colab.research.google.com/drive/1vyN5zo6rqUp_dYNbT4Yrco66zuWCZKoN?usp=sharing)

以下是我带领感兴趣的读者逐步讲解代码的指南。

## 数据集

大型语言模型的燃料是一个文本数据流，可选地分成多个文档集。在生产级应用中，每个文档可能是一个互联网网页，但对于microgpt，我们使用一个更简单的例子——32,000个名字，每行一个：

In [1]:
import os       # os.path.exists
import random   # random.seed, random.choices, random.gauss, random.shuffle
random.seed(42) # 在混沌中建立秩序

# 让输入数据集 `docs`：list[str] 文档列表（例如名字数据集）
if not os.path.exists('input.txt'):
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')
docs = [l.strip() for l in open('input.txt').read().strip().split('\n') if l.strip()] # list[str] 文档列表
random.shuffle(docs)
print(f"num docs: {len(docs)}")

num docs: 32033


数据集看起来像这样，每个名字是一个文档：

```
emma
olivia
ava
isabella
sophia
charlotte
mia
amelia
harper
... (大约32,000个名字)
```

模型的目标是学习数据中的模式，然后生成具有相似统计特征的新文档。提前剧透：脚本运行结束时，我们的模型将生成（『幻觉』！）新的、看似合理的名字。抢先看：

```
sample  1: kamon
sample  2: ann
sample  3: karai
sample  4: jaire
sample  5: vialan
sample  6: karia
sample  7: yeran
sample  8: anna
sample  9: areli
sample 10: kaina
sample 11: konna
sample 12: keylen
sample 13: liole
sample 14: alerin
sample 15: earan
sample 16: lenne
sample 17: kana
sample 18: lara
sample 19: alela
sample 20: anton
```

看起来平平无奇，但从ChatGPT这样的模型角度来看，你与它的对话只是一个看起来有点特别的『文档』。当你用提示初始化文档时，从模型的角度来看，它的响应只是文档的统计式补全。

## 分词器

在后台，神经网络处理数字而不是字符，所以我们需要一种方法将文本转换为整数标记ID序列，反之亦然。像[tiktoken](https://github.com/openai/tiktoken)（由GPT-4使用）这样的生产分词器为了效率在字符块上操作，但最简单的可能的分词器只是为数据集中的每个唯一字符分配一个整数：

In [2]:
# 让有一个分词器将字符串翻译为离散符号并反向翻译
uchars = sorted(set(''.join(docs))) # 数据集中的唯一字符变成标记ID 0..n-1
BOS = len(uchars) # 特殊“序列开始”（BOS）标记的标记ID
vocab_size = len(uchars) + 1 # 总的唯一标记数，+1是BOS
print(f"vocab size: {vocab_size}")

vocab size: 27


在上面的代码中，我们收集数据集中所有唯一字符（即所有小写字母a-z），排序后每个字母按其索引获得一个ID。请注意，整数值本身没有任何意义；每个标记只是一个离散符号。代替0、1、2，它们完全可以是不同的emoji。此外，我们还创建了一个特殊标记`BOS`（序列开始），它充当分隔符：告诉模型『一个新文档在这里开始／结束』。训练期间，每个文档两侧都被`BOS`包裹：`[BOS, e, m, m, a, BOS]`。模型学会`BOS`启动一个新名字，另一个`BOS`结束它。因此，最终词汇表大小为27（26个可能的小写字母a-z加上1个BOS标记）。

## 自动微分

训练神经网络需要梯度：对于模型中的每个参数，我们需要知道『如果我把这个数稍微调高一点，损失会上升还是下降，幅度多大？』。计算图有多个输入（模型参数和输入标记），但汇聚到单个标量输出：损失（我们将在下面精确定义损失是什么）。反向传播从该单个输出开始，沿着图反向工作，计算损失相对于每个输入的梯度。它依赖微积分中的链式法则。在生产中，像PyTorch这样的库会自动处理。这里，我们在一个叫`Value`的类中从头实现：

In [3]:
import math     # math.log, math.exp

class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data                # 在前向传播中计算的此节点的标量值
        self.grad = 0                   # 损失相对于此节点的导数，在反向传播中计算
        self._children = children       # 计算图中此节点的子节点
        self._local_grads = local_grads # 此节点相对于其子节点的局部导数

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, other): return Value(self.data**other, (self,), (other * self.data**(other-1),))
    def log(self): return Value(math.log(self.data), (self,), (1/self.data,))
    def exp(self): return Value(math.exp(self.data), (self,), (math.exp(self.data),))
    def relu(self): return Value(max(0, self.data), (self,), (float(self.data > 0),))
    def __neg__(self): return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad

我意识到这是数学和算法上最密集的部分，我为此录制了一个2.5小时的视频：[micrograd视频](https://www.youtube.com/watch?v=VMj-3S1tku0)。简而言之，`Value`包装单个标量数字（`.data`）并跟踪它是如何被计算出来的。把每个运算想象成一块小乐高积木：它接受一些输入，产生一个输出（前向传播），并且知道它的输出相对于每个输入会如何变化（局部梯度）。这就是自动微分需要每块积木提供的全部信息。其他一切只是链式法则——把积木串在一起。

每次你对`Value`对象进行数学运算（加法、乘法等），结果是一个新的`Value`，它记住其输入（`_children`）和该运算的局部导数（`_local_grads`）。例如，`__mul__`记录$\frac{\partial(a \cdot b)}{\partial a} = b$和$\frac{\partial(a \cdot b)}{\partial b} = a$。完整的乐高积木集：

| 操作 | 前向 | 局部梯度 |
|-----------|---------|----------------|
| `a + b` | $a + b$ | $\frac{\partial}{\partial a} = 1, \quad \frac{\partial}{\partial b} = 1$ |
| `a * b` | $a \cdot b$ | $\frac{\partial}{\partial a} = b, \quad \frac{\partial}{\partial b} = a$ |
| `a ** n` | $a^n$ | $\frac{\partial}{\partial a} = n \cdot a^{n-1}$ |
| `log(a)` | $\ln(a)$ | $\frac{\partial}{\partial a} = \frac{1}{a}$ |
| `exp(a)` | $e^a$ | $\frac{\partial}{\partial a} = e^a$ |
| `relu(a)` | $\max(0, a)$ | $\frac{\partial}{\partial a} = \mathbf{1}_{a > 0}$ |

`backward()`方法以反向拓扑顺序遍历此图（从损失开始，到参数结束），在每一步应用链式法则。如果损失是$L$，节点$v$有一个子节点$c$，局部梯度为$\frac{\partial v}{\partial c}$，那么：

$$\frac{\partial L}{\partial c} \mathrel{+}= \frac{\partial v}{\partial c} \cdot \frac{\partial L}{\partial v}$$

如果你对微积分不太熟悉，这看起来可能有点吓人，但这本质上只是以直观的方式将两个数相乘。可以这样理解：『如果汽车的速度是自行车的两倍，自行车的速度是步行人的四倍，那么汽车的速度就是步行人的2 × 4 = 8倍。』链式法则就是同样的思想：你沿着路径乘以变化率。

我们从在损失节点设置`self.grad = 1`开始，因为$\frac{\partial L}{\partial L} = 1$：损失相对于自身的变化率就是1。从那里开始，链式法则沿着回到参数的每条路径乘以局部梯度。

注意`+=`（累加，不是赋值）。当一个值在图中的多个地方被使用时（即图有分支），梯度沿着每条分支独立流回，必须求和。这是多变量链式法则的结果：如果$c$通过多条路径对$L$有贡献，总导数是来自每条路径的贡献之和。

`backward()`完成后，图中的每个`Value`都有一个`.grad`包含$\frac{\partial L}{\partial v}$，这告诉我们微调该值会使最终损失如何变化。

以下是一个具体例子。注意`a`被使用了两次（图有分支），所以它的梯度是两条路径之和：

```python
a = Value(2.0)
b = Value(3.0)
c = a * b       # c = 6.0
L = c + a       # L = 8.0
L.backward()
print(a.grad)   # 4.0 (dL/da = b + 1 = 3 + 1, 经由两条路径)
print(b.grad)   # 2.0 (dL/db = a = 2)
```

这正是PyTorch的`.backward()`给出的结果：

```python
import torch
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = a * b
L = c + a
L.backward()
print(a.grad)   # tensor(4.)
print(b.grad)   # tensor(2.)
```

这和PyTorch的`loss.backward()`运行的是相同算法——只是在标量上而非张量（标量数组）上运行——算法上完全一致，显著更小更简单，但当然效率低得多。

让我们详细说明`.backward()`的计算结果。自动微分计算出，如果`L = a*b + a`，且`a=2`、`b=3`，那么`a.grad = 4.0`告诉我们`a`对`L`的局部影响。如果你微调输入`a`，`L`会朝什么方向变化？这里`L`对`a`的导数是4.0，意味着如果我们增加`a`一小点（比如0.001），`L`将增加大约4倍（0.004）。同样，`b.grad = 2.0`意味着同样的微调`b`会增加`L`大约2倍（0.002）。换句话说，这些梯度告诉我们每个输入对最终输出（损失）的影响**方向**（正或负取决于符号）和**陡峭程度**（大小）。这让我们能够迭代地微调神经网络参数来降低损失，从而改进模型的预测。

## 参数

参数是模型的知识。它们是一个大集合的浮点数（为自动微分用`Value`包装），从随机开始，在训练期间迭代优化。每个参数的确切角色将在我们下面定义模型架构后更有意义，但现在我们只需要初始化它们：

In [4]:
n_embd = 16     # 嵌入维度
n_head = 4      # 注意力头数
n_layer = 1     # 层数
block_size = 16 # 最大序列长度
head_dim = n_embd // n_head # 每个头的维度
matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]
state_dict = {'wte': matrix(vocab_size, n_embd), 'wpe': matrix(block_size, n_embd), 'lm_head': matrix(vocab_size, n_embd)}
for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)
params = [p for mat in state_dict.values() for row in mat for p in row]
print(f"num params: {len(params)}")

num params: 4192


每个参数从高斯分布中抽取的小随机数初始化。`state_dict`将它们组织成命名矩阵（借用PyTorch的术语）：嵌入表、注意力权重、MLP权重和最终输出投影。我们还将所有参数展平为单个列表`params`，以便优化器稍后遍历它们。在我们的小模型中，总共4,192个参数。GPT-2有16亿，现代LLM有数千亿。

## 架构

模型架构是一个无状态函数：它接收一个标记、一个位置、参数以及来自前面位置的缓存键/值，并返回logits（分数），表示模型认为接下来应该出现哪个标记。我们遵循GPT-2的设计，做了一些简化：用RMSNorm代替LayerNorm，无偏置，用ReLU代替GeLU。首先是三个小辅助函数：

In [5]:
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

`linear`是矩阵-向量乘法。它接收向量`x`和权重矩阵`w`，计算`w`每行的点积。这是神经网络的基本构建块：一个可学习的线性变换。

In [6]:
def softmax(logits):
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

`softmax`将原始分数向量（logits，范围从$-\infty$到$+\infty$）转换为概率分布：转换后所有值在$[0, 1]$范围内且求和为1。我们先减去最大值以保证数值稳定性（数学上不改变结果，但防止`exp`溢出）。

In [7]:
def rmsnorm(x):
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

`rmsnorm`（均方根归一化）将向量重新缩放，使其值的均方根为1。这防止激活值在网络中流动时膨胀或缩小，从而稳定训练。它是原始GPT-2中使用的LayerNorm的更简单变体。

现在看模型本身：

In [8]:
def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id] # 标记嵌入
    pos_emb = state_dict['wpe'][pos_id] # 位置嵌入
    x = [t + p for t, p in zip(tok_emb, pos_emb)] # 联合标记和位置嵌入
    x = rmsnorm(x)

    for li in range(n_layer):
        # 1) 多头注意力块
        x_residual = x
        x = rmsnorm(x)
        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])
        keys[li].append(k)
        values[li].append(v)
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]
            attn_logits = [sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5 for t in range(len(k_h))]
            attn_weights = softmax(attn_logits)
            head_out = [sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h))) for j in range(head_dim)]
            x_attn.extend(head_out)
        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]
        # 2) MLP块
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
        x = [xi.relu() for xi in x]
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])
        x = [a + b for a, b in zip(x, x_residual)]

    logits = linear(x, state_dict['lm_head'])
    return logits

该函数处理一个标记（ID为`token_id`）在特定时间位置（`pos_id`），以及来自之前迭代的一些上下文，这些上下文总结在`keys`和`values`的激活中，称为KV缓存。以下是逐步讲解：

**嵌入。** 神经网络无法直接处理像5这样的原始标记ID。它只能处理向量（数字列表）。所以我们为每个可能的标记关联一个可学习的向量，将其作为神经签名输入。标记ID和位置ID各自从各自的嵌入表（`wte`和`wpe`）中查找一行。这两个向量相加，给模型一个既编码标记是*什么*又编码它在序列中*在哪里*的表示。现代LLM通常跳过位置嵌入，改用其他基于相对位置的方案，例如RoPE。

**注意力块。** 当前标记被投影为三个向量：查询（Q）、键（K）和值（V）。直观地说，查询说『我在找什么？』，键说『我包含什么？』，值说『如果被选中，我会提供什么？』。例如，在名字『emma』中，当模型在第二个『m』并试图预测接下来会出现什么时，它可能学到一个类似『最近出现了哪些元音？』的查询。早前的『e』会有一个与这个查询很好匹配的键，因此它获得高注意力权重，它的值（关于是元音的信息）流入当前位置。键和值被追加到KV缓存中，以便之前的位置可用。每个注意力头计算其查询与所有缓存键之间的点积（按$\sqrt{d_{head}}$缩放），应用softmax获得注意力权重，然后取缓存值的加权和。所有头的输出被拼接并通过`attn_wo`投影。值得强调的是，注意力块是位置`t`处的标记唯一可以『查看』过去`0..t-1`中标记的确切且唯一的地方。注意力是一种**标记通信机制**。

**MLP块。** MLP是『多层感知器』的缩写，它是一个两层前馈网络：先投影到嵌入维度的4倍，应用ReLU，再投影回来。这是模型每个位置进行大部分『思考』的地方。与注意力不同，此计算完全在时间`t`本地。Transformer将通信（注意力）与计算（MLP）穿插。

**残差连接。** 注意力和MLP块都将其输出加回其输入（`x = [a + b for ...]`）。这让梯度直接流过网络，使更深的模型可训练。

**输出。** 最终隐藏状态由`lm_head`投影到词汇表大小，为词汇表中的每个标记产生一个logit。在我们的情况下，只是27个数字。logit越高 = 模型认为对应标记更可能接下来出现。

你可能会注意到我们在训练期间使用了KV缓存，这不太寻常。人们通常将KV缓存仅与推理关联。但KV缓存在概念上始终存在，即使在训练期间也是如此。在生产实现中，它只是隐藏在高度向量化的注意力计算中，该计算同时处理序列中的所有位置。由于microgpt一次处理一个标记（无批处理维度，无并行时间步），我们显式构建KV缓存。与典型的推理设置不同（KV缓存持有分离的张量），这里缓存的键和值是计算图中的活`Value`节点，所以我们实际上通过它们进行反向传播。

## 训练循环

现在我们将所有东西串联起来。训练循环重复：(1)选择一个文档，(2)在其标记上运行模型前向传播，(3)计算损失，(4)反向传播获得梯度，(5)更新参数。

In [9]:
# 让有Adam——被祝福的优化器及其缓冲区
learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m = [0.0] * len(params) # 第一矩缓冲区
v = [0.0] * len(params) # 第二矩缓冲区

# 按顺序重复
num_steps = 1000 # 训练步数
for step in range(num_steps):

    # 取单个文档，分词，用特殊BOS标记在两侧包裹
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    # 将标记序列通过模型前向传播，一路构建到损失的计算图。
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)
    loss = (1 / n) * sum(losses) # 文档序列上的最终平均损失。愿你的损失很低。

    # 反向传播损失，计算相对于所有模型参数的梯度。
    loss.backward()

    # Adam优化器更新：根据相应梯度更新模型参数。
    lr_t = learning_rate * (1 - step / num_steps) # 线性学习率衰减
    for i, p in enumerate(params):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad
        v[i] = beta2 * v[i] + (1 - beta2) * p.grad ** 2
        m_hat = m[i] / (1 - beta1 ** (step + 1))
        v_hat = v[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0

    print(f"step {step+1:4d} / {num_steps:4d} | loss {loss.data:.4f}")

step    1 / 1000 | loss 3.3660
step    2 / 1000 | loss 3.4243
step    3 / 1000 | loss 3.1778
step    4 / 1000 | loss 3.0664
step    5 / 1000 | loss 3.2209
step    6 / 1000 | loss 2.9452
step    7 / 1000 | loss 3.2894
step    8 / 1000 | loss 3.3245
step    9 / 1000 | loss 2.8990
step   10 / 1000 | loss 3.2229
step   11 / 1000 | loss 2.7964
step   12 / 1000 | loss 2.9345
step   13 / 1000 | loss 3.0544
step   14 / 1000 | loss 3.0905
step   15 / 1000 | loss 3.0651
step   16 / 1000 | loss 2.7337
step   17 / 1000 | loss 2.8839
step   18 / 1000 | loss 2.8977
step   19 / 1000 | loss 2.7073
step   20 / 1000 | loss 2.7453
step   21 / 1000 | loss 3.7212
step   22 / 1000 | loss 2.8026
step   23 / 1000 | loss 2.8241
step   24 / 1000 | loss 2.0374
step   25 / 1000 | loss 3.3698
step   26 / 1000 | loss 2.9154
step   27 / 1000 | loss 3.2795
step   28 / 1000 | loss 2.9195
step   29 / 1000 | loss 2.3027
step   30 / 1000 | loss 2.2691
step   31 / 1000 | loss 2.8957
step   32 / 1000 | loss 2.9539
step   3

让我们逐一讲解：

**分词。** 每个训练步骤选择一个文档，并用`BOS`在两侧包裹：名字『emma』变成`[BOS, e, m, m, a, BOS]`。模型的任务是：给定前面的标记，预测每个下一个标记。

**前向传播与损失。** 我们一次一个地将标记馈入模型，随着进行构建KV缓存。在每个位置，模型输出27个logits，我们通过softmax将其转换为概率。每个位置的损失是正确下一个标记的负对数概率：$-\log p(\text{target})$。这被称为交叉熵损失。直观地，损失衡量预测错误程度：模型对实际接下来发生的内容有多惊讶。如果模型给正确标记分配概率1.0，它完全不惊讶，损失为0。如果分配接近0的概率，模型非常惊讶，损失趋向$+\infty$。我们对文档上每个位置的损失取平均，获得单个标量损失。

**反向传播。** 一次调用`loss.backward()`通过整个计算图运行反向传播，从损失一路回到softmax、模型和每个参数。之后，每个参数的`.grad`告诉我们如何改变它以降低损失。

**Adam优化器。** 我们可以直接做`p.data -= lr * p.grad`（梯度下降），但Adam更聪明。它为每个参数维护两个运行平均值：`m`跟踪最近梯度的均值（动量，像一个滚动的球），`v`跟踪最近平方梯度的均值（为每个参数自适应调整学习率）。`m_hat`和`v_hat`是偏差校正，用于解释`m`和`v`被初始化为零且需要预热的事实。学习率在训练过程中线性衰减。更新后，我们为下一步重置`.grad = 0`。

在1,000步上，损失从约3.3（在27个标记中随机猜测：$-\log(1/27) \approx 3.3$）下降到约2.37。越低越好，最低可能为0（完美预测），所以仍有改进空间，但模型显然在学习名字的统计模式。

## 推理

训练完成后，我们可以从模型中采样新名字。参数被冻结，我们只是在循环中运行前向传播，将每个生成的标记反馈为下一个输入：

In [10]:
# Inference: may the model babble back to us
temperature = 0.5 # 在 (0, 1] 中，控制生成文本的『创意』程度，从低到高
print("\n--- 推理（新的、幻觉的名字） ---")
for sample_idx in range(20):
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    token_id = BOS
    sample = []
    for pos_id in range(block_size):
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax([l / temperature for l in logits])
        token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
        if token_id == BOS:
            break
        sample.append(uchars[token_id])
    print(f"sample {sample_idx+1:2d}: {''.join(sample)}")


--- 推理（新的、幻觉的名字） ---
sample  1: kamon
sample  2: ann
sample  3: karai
sample  4: jaire
sample  5: vialan
sample  6: karia
sample  7: yeran
sample  8: anna
sample  9: areli
sample 10: kaina
sample 11: konna
sample 12: keylen
sample 13: liole
sample 14: alerin
sample 15: earan
sample 16: lenne
sample 17: kana
sample 18: lara
sample 19: alela
sample 20: anton


我们用`BOS`标记开始每个样本，它告诉模型『开始一个新名字』。模型产生27个logits，我们将其转换为概率，并根据这些概率随机采样一个标记。那个标记被反馈为下一个输入，我们重复直到模型再次产生`BOS`（意思是『我完成了』）或达到最大序列长度。

`temperature`参数控制随机性。在softmax之前，我们将logits除以温度。温度1.0直接从模型学习的分布中采样。较低的温度（如这里的0.5）锐化分布，使模型更保守，倾向于选择最高概率的选项。接近0的温度将始终选择单个最可能的标记（贪心解码）。较高的温度使分布更平坦，产生更多样但可能不太连贯的输出。

## 运行它

你只需要Python（无需pip install，零依赖）：

```bash
python train.py
```

该脚本在我的MacBook上大约需要1分钟。你会看到每一步打印的损失：

```
train.py
num docs: 32033
vocab size: 27
num params: 4192
step    1 / 1000 | loss 3.3660
step    2 / 1000 | loss 3.4243
step    3 / 1000 | loss 3.1778
step    4 / 1000 | loss 3.0664
step    5 / 1000 | loss 3.2209
step    6 / 1000 | loss 2.9452
step    7 / 1000 | loss 3.2894
step    8 / 1000 | loss 3.3245
step    9 / 1000 | loss 2.8990
step   10 / 1000 | loss 3.2229
step   11 / 1000 | loss 2.7964
step   12 / 1000 | loss 2.9345
step   13 / 1000 | loss 3.0544
...
```

观察它从~3.3（随机水平）下降到~2.37。这个数字越低，网络对序列中接下来会出现什么的预测就越好。训练结束时，训练标记序列的统计模式知识被提炼在模型参数中。固定这些参数，我们现在可以生成新的、幻觉的名字。你会看到（再次）：

```
sample  1: kamon
sample  2: ann
sample  3: karai
sample  4: jaire
sample  5: vialan
sample  6: karia
sample  7: yeran
sample  8: anna
sample  9: areli
sample 10: kaina
sample 11: konna
sample 12: keylen
sample 13: liole
sample 14: alerin
sample 15: earan
sample 16: lenne
sample 17: kana
sample 18: lara
sample 19: alela
sample 20: anton
```

作为在你自己电脑上运行脚本的替代方案，你可以直接在 [Google Colab笔记本](https://colab.research.google.com/drive/1vyN5zo6rqUp_dYNbT4Yrco66zuWCZKoN?usp=sharing) 上运行它，并向Gemini提问相关问题。试着玩一玩这个脚本！你可以尝试不同的数据集，或者训练更长时间（增加`num_steps`），或增大模型尺寸来获得越来越好的结果。

## 渐进式构建

要像剥洋葱一样逐层看代码构建，建议的演进路径大致如下：

| 文件 | 新增内容 |
|------|----------|
| `train0.py` | 二元语法计数表——无神经网络，无梯度 |
| `train1.py` | MLP + 手动梯度（数值与分析）+ SGD |
| `train2.py` | 自动微分（Value类）——替换手动梯度 |
| `train3.py` | 位置嵌入 + 单头注意力 + RMSNorm + 残差连接 |
| `train4.py` | 多头注意力 + 层循环——完整GPT架构 |
| `train5.py` | Adam优化器——这就是 `train.py` |

我创建了一个名为 [build_microgpt.py](https://gist.github.com/karpathy/561ac2de12a47cc06a23691e1be9543a) 的Gist，在Revisions中你可以看到所有这些版本以及每步之间的差异。我认为这可能是逐步浏览代码库的一种有帮助的方式——每次只添加一个组件。

## 现实世界的GPT

microgpt包含了训练和运行GPT的完整算法本质。但在这和像ChatGPT这样的生产级LLM之间，有一长串不同的东西。它们都不改变核心算法和整体布局，但它们是使其在规模上实际运作的关键。按相同章节顺序来看：

**数据。** 代替3.2万个短名字，生产模型在数万亿标记的互联网文本上训练：网页、书籍、代码等。数据被去重、按质量过滤，并在各领域之间精心配比。

**分词器。** 代替单个字符，生产模型使用子词分词器如BPE（字节对编码），这些分词器学会将经常共同出现的字符序列合并为单个标记。常见词如『the』变成单个标记，稀有词被分解为片段。这给出约10万个标记的词汇表，并且因为模型在每个位置看到更多内容而效率高得多。

**自动微分。** microgpt在纯Python中的标量`Value`对象上操作。生产系统使用张量（数字的大型多维数组）并在GPU/TPU上运行，每秒执行数十亿次浮点运算。像PyTorch这样的库在张量上处理自动微分，像FlashAttention这样的CUDA内核融合多个操作以加速。数学完全一样，只是对应许多标量并行处理。

**架构。** microgpt有4,192个参数。GPT-4级别的模型有数千亿。总体上它是一个非常相似的Transformer神经网络，只是宽得多（嵌入维度超过10,000）和深得多（100+层）。现代LLM还融入了一些更多类型的乐高积木并调整了顺序：例子包括RoPE（旋转位置嵌入）替代学习位置嵌入、GQA（分组查询注意力）减少KV缓存大小、门控线性激活替代ReLU、混合专家（MoE）层等。但注意力（通信）和MLP（计算）穿插在残差流上的核心结构被很好地保留。

**训练。** 代替每步一个文档，生产训练使用大批量（每步数百万标记）、梯度累积、混合精度（float16/bfloat16）和谨慎的超参数调优。训练一个前沿模型需要数千个GPU运行数月。

**优化。** microgpt使用Adam和简单的线性学习率衰减，仅此而已。在规模上，优化成为一门独立的学科。模型以降低精度（bfloat16甚至fp8）在大型GPU集群上训练以求效率，这引入了自身的数值挑战。优化器设置（学习率、权重衰减、beta参数、预热计划、衰减计划）必须精确调优，正确的值取决于模型大小、批大小和数据集组成。缩放定律（如Chinchilla）指导如何在固定计算预算下在模型大小和训练标记数之间分配。在规模上将任何这些细节搞错可能浪费数百万美元的计算资源，因此团队在投入完整训练之前会运行大量较小规模的实验来预测正确设置。

**后训练。** 训练出来的基础模型（称为『预训练』模型）是一个文档补全器，不是聊天机器人。将其转变为ChatGPT发生在两个阶段。首先，SFT（监督微调）：你只需将文档替换为策划的对话并继续训练。算法上没有任何改变。其次，RL（强化学习）：模型生成响应，它们被评分（由人、另一个『裁判』模型或算法），模型从该反馈中学习。从根本上说，模型仍在文档上训练，但现在这些文档由来自模型本身的标记组成。

**推理。** 为数百万用户提供模型服务需要自己的工程堆栈：将请求批处理在一起、KV缓存管理和分页（vLLM等）、推测解码以提高速度、量化（以int8/int4运行而不是float16）以减少内存，以及将模型分布到多个GPU。从根本上说，我们仍在预测序列中的下一个标记，但花了大量工程使其更快。

所有这些都是重要的工程和研究贡献，但如果你理解microgpt，你就理解了算法本质。

## 常见问题

**模型『理解』任何东西吗？** 这是一个哲学问题，但从机制上讲：没有魔法在发生。模型是一个大数学函数，将输入标记映射到下一个标记的概率分布。在训练期间，参数被调整以使正确的下一个标记更可能。这是否构成『理解』取决于你，但机制完全包含在上面的200行中。

**它为什么能工作？** 模型有数千个可调参数，优化器每一步将它们微调一点点以使损失下降。经过许多步，参数定居为捕获数据统计规律的值。对于名字，这意味着诸如：名字通常以辅音开头，『qu』倾向于一起出现，名字很少有三个连续辅音等。模型不学习显式规则，它学习一个恰好反映这些规则的概率分布。

**这与ChatGPT有什么关系？** ChatGPT是相同的核心循环（预测下一个标记、采样、重复）大规模扩展，加上后训练使其具有对话能力。当你与它聊天时，系统提示、你的消息和它的回复都只是序列中的标记。模型一次一个标记地补全文档，和microgpt补全名字完全一样。

**『幻觉』是怎么回事？** 模型通过从概率分布采样来生成标记。它没有真理的概念，它只知道给定训练数据，哪些序列在统计上是看似合理的。microgpt『幻觉』一个像『karia』这样的名字，与ChatGPT自信地陈述一个错误事实是相同的现象。两者都是看似合理的补全，恰好不是真实的。

**为什么这么慢？** microgpt在纯Python中一次处理一个标量。单个训练步骤需要几秒钟。相同的数学在GPU上并行处理数百万个标量，速度快几个数量级。

**我能让它生成更好的名字吗？** 可以。训练更长时间（增加`num_steps`），使模型更大（`n_embd`、`n_layer`、`n_head`），或使用更大的数据集。这些是在规模上同样重要的旋钮。

**如果我更换数据集会怎样？** 模型将学习数据中的任何模式。换成城市名、宝可梦名、英文单词或短诗的文件，模型将学习生成那些代替。代码的其余部分不需要改变。